# BS Commitment Attribution Patching with NNsight

Barebones version of the old IOI notebook, but using matched BS commitment branches:

- deceptive target branch: `x_D = p + s_D`
- truthful source branch: `x_H = p + s_H`

Both branches share the same prefix `p`, and commitment examples are filtered by
`Delta_k > 0.3`.

The intervention is one-directional: run the truthful branch as the source, run the
deceptive branch as the target, and patch truthful activations into the deceptive run.
The default patching metric is `-score_C(s_D | p)`, so higher means the intervention
reduced model support for the deceptive commitment sentence. The fixed truthful score
`score(s_H | p)` is still reported for sanity checks.

This notebook keeps the `nnsight` flow simple and only does attribution patching over
attention-head outputs.


In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import importlib
import inspect
import json
import math
import re
import sys
from pathlib import Path

import einops
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import torch
import torch.nn.functional as F
from IPython.display import Markdown, clear_output, display
import nnsight
from nnsight import LanguageModel

REPO_ROOT = Path("/playpen-ssd/smerrill/deception2")
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import activation_patching as ap
from activation_patching import encode_text_for_model, resolve_decoder_layers
import activation_patching_debug as apd
apd = importlib.reload(apd)

pio.renderers.default = "plotly_mimetype+notebook_connected+notebook"
pd.options.display.max_colwidth = 160
print("nnsight", nnsight.__version__)


In [ ]:
def latest_snapshot_path(root: Path) -> Path | None:
    snapshot_root = root / "snapshots"
    if not snapshot_root.exists():
        return None
    snapshots = sorted(path for path in snapshot_root.iterdir() if path.is_dir())
    return snapshots[-1] if snapshots else None


MODEL_ID = os.environ.get("ATTR_PATCH_MODEL_ID", "gpt-oss-20b").strip()
MODEL_CONFIGS = {
    "gpt-oss-20b": {
        "hf_repo": "openai/gpt-oss-20b",
        "hf_cache_dir": "models--openai--gpt-oss-20b",
    },
    "DeepSeek-R1-Distill-Qwen-7B": {
        "hf_repo": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
        "hf_cache_dir": "models--deepseek-ai--DeepSeek-R1-Distill-Qwen-7B",
    },
}
if MODEL_ID not in MODEL_CONFIGS:
    raise ValueError(f"Unsupported ATTR_PATCH_MODEL_ID={MODEL_ID!r}. Choose from {sorted(MODEL_CONFIGS)}.")

model_cfg = MODEL_CONFIGS[MODEL_ID]
hf_cache_root = Path("/playpen-ssd/smerrill/huggingface/transformers")
cached_snapshot = latest_snapshot_path(hf_cache_root / model_cfg["hf_cache_dir"])
LOCALIZATION_DIR = Path(
    os.environ.get(
        "ATTR_PATCH_LOCALIZATION_DIR",
        str(REPO_ROOT / "DatasetMain" / "bs" / MODEL_ID / "localization"),
    )
)
LOCAL_MODEL_SNAPSHOT = Path(
    os.environ.get(
        "ATTR_PATCH_MODEL_SNAPSHOT",
        str(cached_snapshot or model_cfg["hf_repo"]),
    )
)
MODEL_NAME = str(
    LOCAL_MODEL_SNAPSHOT if LOCAL_MODEL_SNAPSHOT.exists() else model_cfg["hf_repo"]
)
PAIR_CACHE_PATH = Path(
    os.environ.get(
        "ATTR_PATCH_PAIR_CACHE_PATH",
        str(REPO_ROOT / "Cache" / "activation_patching" / f"bs_commitment_pairs_for_notebook__{MODEL_ID}.jsonl"),
    )
)
DTYPE_BY_NAME = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}
DTYPE_NAME = os.environ.get("ATTR_PATCH_DTYPE", "bfloat16").strip().lower()
if DTYPE_NAME not in DTYPE_BY_NAME:
    raise ValueError(f"Unsupported ATTR_PATCH_DTYPE={DTYPE_NAME!r}. Choose from {sorted(DTYPE_BY_NAME)}.")
MODEL_DTYPE = DTYPE_BY_NAME[DTYPE_NAME]
PAIR_COUNT = max(50, int(os.environ.get("ATTR_PATCH_PAIR_COUNT", "50")))
BATCH_PAIR_COUNT = int(os.environ.get("ATTR_PATCH_BATCH_PAIR_COUNT", "1"))
PAIR_SEARCH_LIMIT = int(os.environ.get("ATTR_PATCH_PAIR_SEARCH_LIMIT", "128"))
PATCH_METRIC_NAME = "-score_C(s_D | p)"
PATCH_SCOPE_ALIASES = {
    "commitment-first": "commitment_first",
    "commitment_first": "commitment_first",
    "commitment-token-first": "commitment_first",
    "commitment_token_first": "commitment_first",
    "first-commitment": "commitment_first",
    "first_commitment": "commitment_first",
    "first-token": "commitment_first",
    "first_token": "commitment_first",
    "commitment-first-n": "commitment_first_n",
    "commitment_first_n": "commitment_first_n",
    "commitment-first-x": "commitment_first_n",
    "commitment_first_x": "commitment_first_n",
    "first-n": "commitment_first_n",
    "first_n": "commitment_first_n",
    "first-n-tokens": "commitment_first_n",
    "first_n_tokens": "commitment_first_n",
    "prefix-final": "commitment_first",
    "prefix_final": "commitment_first",
    "prefix": "commitment_first",
    "commitment-sentence-mean": "commitment_mean",
    "commitment_sentence_mean": "commitment_mean",
    "commitment-mean": "commitment_mean",
    "commitment_mean": "commitment_mean",
    "sentence_mean": "commitment_mean",
    "full-sequence": "full_sequence",
    "full_sequence": "full_sequence",
    "full": "full_sequence",
}
PATCH_SCOPE_RAW = os.environ.get("ATTR_PATCH_SCOPE", "commitment_first").strip().lower()
PATCH_SCOPE = PATCH_SCOPE_ALIASES.get(PATCH_SCOPE_RAW, PATCH_SCOPE_RAW)
PATCH_SCOPES = {"commitment_first", "commitment_first_n", "commitment_mean", "full_sequence"}
if PATCH_SCOPE not in PATCH_SCOPES:
    raise ValueError(f"ATTR_PATCH_SCOPE must be one of {sorted(PATCH_SCOPES)}.")
PATCH_FIRST_N_TOKENS = int(os.environ.get("ATTR_PATCH_FIRST_N_TOKENS", "1"))
if PATCH_FIRST_N_TOKENS <= 0:
    raise ValueError("ATTR_PATCH_FIRST_N_TOKENS must be positive.")
SENTENCE_SCORE_MODE = os.environ.get("ATTR_PATCH_SENTENCE_SCORE", "mean_logprob").strip().lower()
SENTENCE_SCORE_MODES = {"mean_logprob", "sum_logprob", "geomean_prob", "sentence_prob"}
if SENTENCE_SCORE_MODE not in SENTENCE_SCORE_MODES:
    raise ValueError(f"ATTR_PATCH_SENTENCE_SCORE must be one of {sorted(SENTENCE_SCORE_MODES)}.")
MIN_COMMITMENT_DELTA = float(os.environ.get("ATTR_PATCH_MIN_COMMITMENT_DELTA", "0.3"))
MIN_COMMITMENT_DECEPTION_RATE = float(os.environ.get("ATTR_PATCH_MIN_COMMITMENT_DECEPTION_RATE", "0.0"))
MIN_DONOR_CLARITY_SCORE = float(os.environ.get("ATTR_PATCH_MIN_DONOR_CLARITY_SCORE", "0.0"))
MIN_NUM_VALID = int(os.environ.get("ATTR_PATCH_MIN_NUM_VALID", "11"))
MIN_SENTENCE_ALPHA_WORDS = int(os.environ.get("ATTR_PATCH_MIN_SENTENCE_ALPHA_WORDS", "4"))
EXCLUDE_MULTILINE_SENTENCES = os.environ.get("ATTR_PATCH_EXCLUDE_MULTILINE_SENTENCES", "1") == "1"
MAX_INPUT_TOKENS_RAW = os.environ.get("ATTR_PATCH_MAX_INPUT_TOKENS", "auto").strip()
if MAX_INPUT_TOKENS_RAW.lower() in {"", "auto", "dataset", "max", "none"}:
    MAX_INPUT_TOKENS = None
else:
    MAX_INPUT_TOKENS = int(MAX_INPUT_TOKENS_RAW)


In [ ]:
model = LanguageModel(
    MODEL_NAME,
    device_map="auto",
    dispatch=True,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
clear_output()
print(model)

tokenizer = model.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

layers, layer_path = resolve_decoder_layers(model)
n_layers = len(layers)
n_heads = int(model.config.num_attention_heads)
head_dim = int(getattr(model.config, "head_dim", 0) or (model.config.hidden_size // n_heads))


def attn_out_input(layer):
    if hasattr(layer, "self_attn") and hasattr(layer.self_attn, "o_proj"):
        return layer.self_attn.o_proj.input
    if hasattr(layer, "attn") and hasattr(layer.attn, "c_proj"):
        return layer.attn.c_proj.input
    raise AttributeError("Could not find the attention output projection input for this layer.")


print(layer_path, "|", n_layers, "layers |", n_heads, "heads")
display(
    Markdown(
        "\n".join(
            [
                f"- `MODEL_ID`: `{MODEL_ID}`",
                f"- `MODEL_NAME`: `{MODEL_NAME}`",
                f"- `MODEL_DTYPE`: `{DTYPE_NAME}`",
                f"- `PATCH_METRIC_NAME`: `{PATCH_METRIC_NAME}`",
                f"- `PATCH_SCOPE`: `{PATCH_SCOPE}`",
                f"- `PATCH_FIRST_N_TOKENS`: `{PATCH_FIRST_N_TOKENS}`",
                f"- `SENTENCE_SCORE_MODE`: `{SENTENCE_SCORE_MODE}`",
                f"- `PAIR_COUNT`: `{PAIR_COUNT}`",
                f"- `BATCH_PAIR_COUNT`: `{BATCH_PAIR_COUNT}`",
                f"- `MIN_NUM_VALID`: `{MIN_NUM_VALID}`",
                f"- `MIN_SENTENCE_ALPHA_WORDS`: `{MIN_SENTENCE_ALPHA_WORDS}`",
                f"- `EXCLUDE_MULTILINE_SENTENCES`: `{EXCLUDE_MULTILINE_SENTENCES}`",
                f"- `MAX_INPUT_TOKENS_RAW`: `{MAX_INPUT_TOKENS_RAW}`",
                f"- `LOCALIZATION_DIR`: `{LOCALIZATION_DIR}`",
                f"- `PAIR_CACHE_PATH`: `{PAIR_CACHE_PATH}`",
                f"- `PYTORCH_CUDA_ALLOC_CONF`: `{os.environ.get('PYTORCH_CUDA_ALLOC_CONF', '(unset)')}`",
            ]
        )
    )
)


In [ ]:
load_commitment_pairs = apd.load_commitment_pairs
load_commitment_pairs_kwargs = dict(
    localization_dir=LOCALIZATION_DIR,
    pair_cache_path=PAIR_CACHE_PATH,
    pair_count=PAIR_COUNT,
    pair_search_limit=PAIR_SEARCH_LIMIT,
    refresh_cache=False,
    min_commitment_delta=MIN_COMMITMENT_DELTA,
    min_commitment_deception_rate=MIN_COMMITMENT_DECEPTION_RATE,
    min_donor_clarity_score=MIN_DONOR_CLARITY_SCORE,
    disable_tqdm=False,
)
load_commitment_pairs_signature = inspect.signature(load_commitment_pairs)
if "min_num_valid" in load_commitment_pairs_signature.parameters:
    load_commitment_pairs_kwargs["min_num_valid"] = MIN_NUM_VALID
if "min_sentence_alpha_words" in load_commitment_pairs_signature.parameters:
    load_commitment_pairs_kwargs["min_sentence_alpha_words"] = MIN_SENTENCE_ALPHA_WORDS
if "exclude_multiline_sentences" in load_commitment_pairs_signature.parameters:
    load_commitment_pairs_kwargs["exclude_multiline_sentences"] = EXCLUDE_MULTILINE_SENTENCES

pairs_df = load_commitment_pairs(**load_commitment_pairs_kwargs).copy()

display(
    pairs_df[
        [
            "pair_index",
            "example_id",
            "commitment_delta",
            "shared_context_num_valid",
            "deceptive_prefix_num_valid",
            "donor_clarity_score",
            "n_truthful_donors",
            "deceptive_commitment_sentence",
            "truthful_donor_sentence",
        ]
    ]
)


In [ ]:
def encode_branch(prefix_text, full_text):
    prefix_ids = encode_text_for_model(
        tokenizer,
        prefix_text,
        max_input_tokens=MAX_INPUT_TOKENS,
    )["input_ids"][0]
    full_ids = encode_text_for_model(
        tokenizer,
        full_text,
        max_input_tokens=MAX_INPUT_TOKENS,
    )["input_ids"][0]
    prefix_len = int(prefix_ids.shape[0])
    total_len = int(full_ids.shape[0])
    if total_len <= prefix_len:
        raise ValueError("Each branch needs at least one token after the shared prefix.")
    if not torch.equal(full_ids[:prefix_len], prefix_ids):
        raise ValueError("Full branch tokenization does not start with the shared prefix tokens.")
    return {
        "full_ids": full_ids,
        "prefix_len": prefix_len,
        "total_len": total_len,
        "score_start_pos": prefix_len,
        "score_stop_pos": total_len,
    }


def prepare_pair_records(pairs):
    prepared = []
    for pair_row in pairs.to_dict(orient="records"):
        deceptive_encoded = encode_branch(
            pair_row["shared_prefix_text"],
            pair_row["deceptive_branch_text"],
        )
        truthful_encoded = encode_branch(
            pair_row["shared_prefix_text"],
            pair_row["truthful_branch_text"],
        )
        if int(deceptive_encoded["prefix_len"]) != int(truthful_encoded["prefix_len"]):
            raise ValueError("Prefix token lengths differ across deceptive/truthful branches.")
        prepared.append(
            {
                **pair_row,
                "deceptive_encoded": deceptive_encoded,
                "truthful_encoded": truthful_encoded,
                "prefix_token_len": int(deceptive_encoded["prefix_len"]),
                "deceptive_total_len": int(deceptive_encoded["total_len"]),
                "truthful_total_len": int(truthful_encoded["total_len"]),
                "max_total_len": int(
                    max(
                        deceptive_encoded["total_len"],
                        truthful_encoded["total_len"],
                    )
                ),
            }
        )
    return sorted(
        prepared,
        key=lambda row: (int(row["max_total_len"]), int(row["pair_index"])),
    )


def pair_chunk_slices(prepared_pairs, batch_pair_count):
    if int(batch_pair_count) <= 0:
        raise ValueError("BATCH_PAIR_COUNT must be positive.")
    return [
        prepared_pairs[start : start + int(batch_pair_count)]
        for start in range(0, len(prepared_pairs), int(batch_pair_count))
    ]


def make_row(prepared_pair, branch_role):
    if branch_role == "deceptive":
        encoded = prepared_pair["deceptive_encoded"]
        sentence_text = prepared_pair["deceptive_commitment_sentence"]
    elif branch_role == "truthful":
        encoded = prepared_pair["truthful_encoded"]
        sentence_text = prepared_pair["truthful_donor_sentence"]
    else:
        raise ValueError(f"Unsupported branch_role={branch_role!r}")
    return {
        "pair_index": int(prepared_pair["pair_index"]),
        "example_id": str(prepared_pair["example_id"]),
        "branch_role": branch_role,
        "sentence_text": sentence_text,
        **encoded,
    }


def build_rows(prepared_pairs):
    source_rows = []
    target_rows = []
    for prepared_pair in prepared_pairs:
        source_rows.append(make_row(prepared_pair, "truthful"))
        target_rows.append(make_row(prepared_pair, "deceptive"))
    return source_rows, target_rows


def pad_rows(rows, *, max_len=None):
    if max_len is None:
        max_len = max(int(row["total_len"]) for row in rows)
    input_ids = torch.full((len(rows), int(max_len)), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(rows), int(max_len)), dtype=torch.long)
    for row_idx, row in enumerate(rows):
        total_len = int(row["total_len"])
        input_ids[row_idx, :total_len] = row["full_ids"]
        attention_mask[row_idx, :total_len] = 1
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "rows": rows,
    }


def build_source_target_batches(
    source_pairs,
    target_pairs,
    *,
    source_role="truthful",
    target_role="deceptive",
):
    if len(source_pairs) != len(target_pairs):
        raise ValueError("source_pairs and target_pairs must have the same batch size.")
    source_rows = [make_row(prepared_pair, source_role) for prepared_pair in source_pairs]
    target_rows = [make_row(prepared_pair, target_role) for prepared_pair in target_pairs]
    max_len = max(
        max(int(row["total_len"]) for row in source_rows),
        max(int(row["total_len"]) for row in target_rows),
    )
    source_batch = pad_rows(source_rows, max_len=max_len)
    target_batch = pad_rows(target_rows, max_len=max_len)
    source_inputs = {
        "input_ids": source_batch["input_ids"],
        "attention_mask": source_batch["attention_mask"],
    }
    target_inputs = {
        "input_ids": target_batch["input_ids"],
        "attention_mask": target_batch["attention_mask"],
    }
    return source_batch, target_batch, source_inputs, target_inputs


def build_batches_for_pairs(prepared_pairs):
    return build_source_target_batches(prepared_pairs, prepared_pairs)


prepared_pairs = prepare_pair_records(pairs_df)
DATASET_MAX_INPUT_TOKENS = max(int(pair["max_total_len"]) for pair in prepared_pairs)
EFFECTIVE_MAX_INPUT_TOKENS = (
    int(DATASET_MAX_INPUT_TOKENS) if MAX_INPUT_TOKENS is None else int(MAX_INPUT_TOKENS)
)
MAX_INPUT_TOKENS = int(EFFECTIVE_MAX_INPUT_TOKENS)
pair_chunks = pair_chunk_slices(prepared_pairs, BATCH_PAIR_COUNT)
total_pairs = len(prepared_pairs)

pairs_overview_df = pd.DataFrame(
    [
        {
            "pair_index": int(pair["pair_index"]),
            "example_id": str(pair["example_id"]),
            "batch_index": int(batch_idx),
            "prefix_token_len": int(pair["prefix_token_len"]),
            "deceptive_total_len": int(pair["deceptive_total_len"]),
            "truthful_total_len": int(pair["truthful_total_len"]),
            "max_total_len": int(pair["max_total_len"]),
            "shared_context_num_valid": int(pair["shared_context_num_valid"]),
            "deceptive_prefix_num_valid": int(pair["deceptive_prefix_num_valid"]),
            "commitment_delta": float(pair["commitment_delta"]),
        }
        for batch_idx, chunk in enumerate(pair_chunks)
        for pair in chunk
    ]
).sort_values("pair_index").reset_index(drop=True)

display(
    pairs_overview_df[
        [
            "pair_index",
            "example_id",
            "batch_index",
            "prefix_token_len",
            "deceptive_total_len",
            "truthful_total_len",
            "max_total_len",
            "shared_context_num_valid",
            "deceptive_prefix_num_valid",
            "commitment_delta",
        ]
    ]
)

preview_source_batch, preview_target_batch, _, _ = build_batches_for_pairs(pair_chunks[0])
preview_rows = []
for batch_name, batch in [("source_truthful", preview_source_batch), ("target_deceptive", preview_target_batch)]:
    for row_idx, row in enumerate(batch["rows"]):
        preview_rows.append(
            {
                "batch": batch_name,
                "row_idx": row_idx,
                "pair_index": row["pair_index"],
                "branch_role": row["branch_role"],
                "prefix_len": row["prefix_len"],
                "total_len": row["total_len"],
                "sentence_text": row["sentence_text"],
            }
        )
display(pd.DataFrame(preview_rows))
print(
    f"Prepared {total_pairs} pair(s) across {len(pair_chunks)} chunk(s) "
    f"with batch_pair_count={BATCH_PAIR_COUNT}."
)
print(
    f"Effective max input tokens: {MAX_INPUT_TOKENS} "
    f"(dataset max needed: {DATASET_MAX_INPUT_TOKENS})"
)


In [ ]:
def clear_memory():
    if hasattr(model, "clear_edits"):
        model.clear_edits()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def saved_value(x):
    return getattr(x, "value", x)


def scored_logits_and_targets(logits, batch):
    scored_logits = []
    scored_targets = []
    row_lengths = []
    for row_idx, row in enumerate(batch["rows"]):
        start = int(row["score_start_pos"])
        stop = int(row["score_stop_pos"])
        row_logits = logits[row_idx, start - 1 : stop - 1, :]
        row_targets = batch["input_ids"][row_idx, start:stop].to(logits.device)
        scored_logits.append(row_logits)
        scored_targets.append(row_targets)
        row_lengths.append(int(stop - start))
    return torch.cat(scored_logits, dim=0), torch.cat(scored_targets, dim=0), row_lengths


def score_sentence_token_log_probs(row_token_log_probs):
    row_token_log_probs = row_token_log_probs.float()
    if SENTENCE_SCORE_MODE == "mean_logprob":
        return row_token_log_probs.mean()
    if SENTENCE_SCORE_MODE == "sum_logprob":
        return row_token_log_probs.sum()
    if SENTENCE_SCORE_MODE == "geomean_prob":
        return torch.exp(row_token_log_probs.mean())
    if SENTENCE_SCORE_MODE == "sentence_prob":
        return torch.exp(row_token_log_probs.sum())
    raise ValueError(f"Unsupported SENTENCE_SCORE_MODE={SENTENCE_SCORE_MODE!r}.")


def sentence_score_by_row(logits, batch):
    flat_logits, flat_targets, row_lengths = scored_logits_and_targets(logits, batch)
    token_log_probs = -F.cross_entropy(
        flat_logits,
        flat_targets,
        reduction="none",
    )
    scores = []
    offset = 0
    for row_len in row_lengths:
        row_token_log_probs = token_log_probs[offset : offset + row_len]
        scores.append(score_sentence_token_log_probs(row_token_log_probs))
        offset += row_len
    return torch.stack(scores)


def target_metric_from_deceptive_scores(deceptive_scores):
    return -deceptive_scores.mean()


def target_metric_from_logits(logits, target_batch):
    target_scores = sentence_score_by_row(logits, target_batch)
    return target_metric_from_deceptive_scores(target_scores)


summary_records = []
source_score_total = 0.0
target_score_total = 0.0
target_metric_total = 0.0
truth_minus_deceptive_total = 0.0

for chunk_idx, chunk_pairs in enumerate(pair_chunks, start=1):
    source_batch, target_batch, source_inputs, target_inputs = build_batches_for_pairs(chunk_pairs)

    with torch.inference_mode():
        source_logits = model.trace(
            source_inputs,
            trace=False,
        ).logits
    source_scores = sentence_score_by_row(source_logits, source_batch).detach().cpu()
    del source_logits
    clear_memory()

    with torch.inference_mode():
        target_logits = model.trace(
            target_inputs,
            trace=False,
        ).logits
    target_scores = sentence_score_by_row(target_logits, target_batch).detach().cpu()
    del target_logits
    clear_memory()

    chunk_pair_count = len(chunk_pairs)
    chunk_source_score = float(source_scores.mean().item())
    chunk_target_score = float(target_scores.mean().item())
    chunk_target_metric = float((-target_scores).mean().item())
    chunk_truth_minus_deceptive = float((source_scores - target_scores).mean().item())
    source_score_total += chunk_source_score * float(chunk_pair_count)
    target_score_total += chunk_target_score * float(chunk_pair_count)
    target_metric_total += chunk_target_metric * float(chunk_pair_count)
    truth_minus_deceptive_total += chunk_truth_minus_deceptive * float(chunk_pair_count)

    for local_idx, pair_row in enumerate(chunk_pairs):
        score_h = float(source_scores[local_idx].item())
        score_d = float(target_scores[local_idx].item())
        summary_records.append(
            {
                "pair_index": int(pair_row["pair_index"]),
                "example_id": str(pair_row["example_id"]),
                "commitment_delta": float(pair_row["commitment_delta"]),
                "max_total_len": int(pair_row["max_total_len"]),
                "score_H_source": score_h,
                "score_D_target": score_d,
                "truthful_minus_deceptive": score_h - score_d,
                "target_metric_neg_score_D": -score_d,
                "deceptive_commitment_sentence": str(pair_row["deceptive_commitment_sentence"]),
                "truthful_donor_sentence": str(pair_row["truthful_donor_sentence"]),
            }
        )

    print(
        f"Scored chunk {chunk_idx}/{len(pair_chunks)} | pairs={chunk_pair_count} | "
        f"max_total_len={max(int(pair['max_total_len']) for pair in chunk_pairs)}"
    )

    del source_scores, target_scores, source_batch, target_batch, source_inputs, target_inputs
    clear_memory()

SOURCE_TRUTHFUL_SCORE = source_score_total / float(total_pairs)
TARGET_DECEPTIVE_SCORE = target_score_total / float(total_pairs)
TARGET_BASELINE = target_metric_total / float(total_pairs)
TRUTH_MINUS_DECEPTIVE_BASELINE = truth_minus_deceptive_total / float(total_pairs)


def teacher_forced_metric(logits, target_batch):
    return target_metric_from_logits(logits, target_batch)


def first_commitment_token_span(source_row, target_row, max_tokens):
    source_start = int(source_row["score_start_pos"])
    source_stop = int(source_row["score_stop_pos"])
    target_start = int(target_row["score_start_pos"])
    target_stop = int(target_row["score_stop_pos"])
    span_len = min(
        int(max_tokens),
        source_stop - source_start,
        target_stop - target_start,
    )
    return source_start, target_start, span_len


def activation_delta_for_patch_scope(source, target, source_rows, target_rows):
    if PATCH_SCOPE == "full_sequence":
        return source - target

    delta = torch.zeros_like(target)
    if PATCH_SCOPE in {"commitment_first", "commitment_first_n"}:
        max_tokens = 1 if PATCH_SCOPE == "commitment_first" else PATCH_FIRST_N_TOKENS
        for row_idx, (source_row, target_row) in enumerate(zip(source_rows, target_rows)):
            source_start, target_start, span_len = first_commitment_token_span(
                source_row,
                target_row,
                max_tokens,
            )
            if span_len <= 0:
                continue
            delta[row_idx : row_idx + 1, target_start : target_start + span_len, :] = (
                source[row_idx : row_idx + 1, source_start : source_start + span_len, :]
                - target[row_idx : row_idx + 1, target_start : target_start + span_len, :]
            )
        return delta

    if PATCH_SCOPE == "commitment_mean":
        for row_idx, (source_row, target_row) in enumerate(zip(source_rows, target_rows)):
            source_start = int(source_row["score_start_pos"])
            source_stop = int(source_row["score_stop_pos"])
            target_start = int(target_row["score_start_pos"])
            target_stop = int(target_row["score_stop_pos"])
            source_vec = source[row_idx : row_idx + 1, source_start:source_stop, :].mean(
                dim=1,
                keepdim=True,
            )
            delta[row_idx : row_idx + 1, target_start:target_stop, :] = (
                source_vec - target[row_idx : row_idx + 1, target_start:target_stop, :]
            )
        return delta

    raise ValueError(f"Unsupported PATCH_SCOPE={PATCH_SCOPE!r}.")


def apply_scoped_head_patch(current, source, source_rows, target_rows, heads):
    for head_idx in heads:
        start = int(head_idx) * head_dim
        stop = start + head_dim
        if PATCH_SCOPE == "full_sequence":
            current[:, :, start:stop] = source[:, :, start:stop]
        elif PATCH_SCOPE in {"commitment_first", "commitment_first_n"}:
            max_tokens = 1 if PATCH_SCOPE == "commitment_first" else PATCH_FIRST_N_TOKENS
            for row_idx, (source_row, target_row) in enumerate(zip(source_rows, target_rows)):
                source_start, target_start, span_len = first_commitment_token_span(
                    source_row,
                    target_row,
                    max_tokens,
                )
                if span_len <= 0:
                    continue
                current[row_idx : row_idx + 1, target_start : target_start + span_len, start:stop] = source[
                    row_idx : row_idx + 1,
                    source_start : source_start + span_len,
                    start:stop,
                ]
        elif PATCH_SCOPE == "commitment_mean":
            for row_idx, (source_row, target_row) in enumerate(zip(source_rows, target_rows)):
                source_start = int(source_row["score_start_pos"])
                source_stop = int(source_row["score_stop_pos"])
                target_start = int(target_row["score_start_pos"])
                target_stop = int(target_row["score_stop_pos"])
                source_vec = source[
                    row_idx : row_idx + 1,
                    source_start:source_stop,
                    start:stop,
                ].mean(dim=1, keepdim=True)
                current[row_idx : row_idx + 1, target_start:target_stop, start:stop] = source_vec.expand(
                    1,
                    target_stop - target_start,
                    head_dim,
                )
        else:
            raise ValueError(f"Unsupported PATCH_SCOPE={PATCH_SCOPE!r}.")


summary_df = pd.DataFrame(summary_records).sort_values("pair_index").reset_index(drop=True)
summary_df["target_metric_positive_is_less_deceptive"] = summary_df["target_metric_neg_score_D"]

metric_sanity_df = pd.DataFrame(
    [
        {
            "n_pairs": int(len(summary_df)),
            "patch_metric": PATCH_METRIC_NAME,
            "sentence_score_mode": SENTENCE_SCORE_MODE,
            "mean_score_H_source": float(summary_df["score_H_source"].mean()),
            "mean_score_D_target": float(summary_df["score_D_target"].mean()),
            "mean_truthful_minus_deceptive": float(summary_df["truthful_minus_deceptive"].mean()),
            "median_truthful_minus_deceptive": float(summary_df["truthful_minus_deceptive"].median()),
            "target_baseline_b_prime": float(TARGET_BASELINE),
            "fixed_truthful_score_reference": float(SOURCE_TRUTHFUL_SCORE),
            "target_deceptive_score_reference": float(TARGET_DECEPTIVE_SCORE),
            "truth_minus_deceptive_baseline": float(TRUTH_MINUS_DECEPTIVE_BASELINE),
        }
    ]
)

display(
    summary_df[
        [
            "pair_index",
            "example_id",
            "commitment_delta",
            "score_H_source",
            "score_D_target",
            "truthful_minus_deceptive",
            "target_metric_neg_score_D",
            "deceptive_commitment_sentence",
            "truthful_donor_sentence",
        ]
    ]
)
display(metric_sanity_df)

print(f"Fixed truthful source score E[score(s_H | p)]: {SOURCE_TRUTHFUL_SCORE:.4f}")
print(f"Unpatched deceptive target score E[score(s_D | p)]: {TARGET_DECEPTIVE_SCORE:.4f}")
print(f"Target baseline b_prime = -E[score(s_D | p)]: {TARGET_BASELINE:.4f}")
print(f"Truth-minus-deceptive reference: {TRUTH_MINUS_DECEPTIVE_BASELINE:.4f}")


In [ ]:
layer_sums = [torch.zeros(n_heads, dtype=torch.float32) for _ in range(n_layers)]
diag_records = []

print(f"Attribution objective: {PATCH_SCOPE} patch truthful source activations into deceptive target and maximize -score_C(s_D | p).")

for chunk_idx, chunk_pairs in enumerate(pair_chunks, start=1):
    source_batch, target_batch, source_inputs, target_inputs = build_batches_for_pairs(chunk_pairs)
    chunk_pair_count = len(chunk_pairs)
    chunk_max_total_len = max(int(pair["max_total_len"]) for pair in chunk_pairs)
    print(
        f"Chunk {chunk_idx}/{len(pair_chunks)} | pairs={chunk_pair_count} | "
        f"max_total_len={chunk_max_total_len}"
    )

    for layer_idx, layer in enumerate(layers):
        clear_memory()

        with model.trace(source_inputs):
            source_proxy = attn_out_input(layer)
            source_out = source_proxy.save()

        with model.trace(target_inputs):
            target_proxy = attn_out_input(layer)
            target_out = target_proxy.save()
            target_grad = target_proxy.grad.save()
            logits = model.lm_head.output
            value = teacher_forced_metric(logits, target_batch)
            traced_value = value.save()
            value.backward()

        patch_delta = activation_delta_for_patch_scope(
            saved_value(source_out),
            saved_value(target_out),
            source_batch["rows"],
            target_batch["rows"],
        )
        layer_attr = einops.reduce(
            saved_value(target_grad) * patch_delta,
            "batch pos (head d_head) -> head",
            "sum",
            head=n_heads,
            d_head=head_dim,
        )
        objective_value = float(saved_value(traced_value).item())
        grad_norm = float(saved_value(target_grad).float().norm().item())
        attr_abs_sum = float(layer_attr.float().abs().sum().item())
        attr_max_abs = float(layer_attr.float().abs().max().item())

        layer_sums[layer_idx] += layer_attr.detach().float().cpu()
        diag_records.append(
            {
                "chunk_index": int(chunk_idx - 1),
                "layer": int(layer_idx),
                "chunk_pairs": int(chunk_pair_count),
                "objective_value": objective_value,
                "grad_norm": grad_norm,
                "attr_abs_sum": attr_abs_sum,
                "attr_max_abs": attr_max_abs,
            }
        )
        print(
            f"  Layer {layer_idx:02d} | obj={objective_value:.4f} | "
            f"grad_norm={grad_norm:.4f} | attr_abs_sum={attr_abs_sum:.4f}"
        )

        del source_out, target_out, target_grad, traced_value, value, patch_delta, layer_attr
        clear_memory()

    del source_batch, target_batch, source_inputs, target_inputs
    clear_memory()

patching_results = torch.stack(
    [layer_sum / float(total_pairs) for layer_sum in layer_sums]
).float().numpy()
diagnostics_df = pd.DataFrame(diag_records)
layer_summary_df = (
    diagnostics_df.groupby("layer", as_index=False)[["grad_norm", "attr_abs_sum", "attr_max_abs"]]
    .mean()
    .sort_values("layer")
    .reset_index(drop=True)
)
display(layer_summary_df)
display(diagnostics_df.head(20))

fig = px.imshow(
    patching_results,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
    title="Truthful-source to deceptive-target attribution patching over attention heads",
    labels={"x": "Head", "y": "Layer", "color": "Attribution"},
)
fig.show()


## Circuit validation

Now use actual one-directional interventions on the attention-head circuit found above.

Source run: truthful branch `x_H = p + s_H`.
Target run: deceptive branch `x_D = p + s_D`.
The intervention patches truthful-source head activations into the deceptive-target run. `ATTR_PATCH_SCOPE` controls whether this is `commitment_first` (default; first token of the commitment sentence), `commitment_first_n` (first `ATTR_PATCH_FIRST_N_TOKENS` commitment tokens), `commitment_mean`, or `full_sequence`.

The patched metric is `m_C = -score_C(s_D | p)`, so larger means the patched run assigns
less support to the deceptive commitment sentence. Define
`Delta(C) = m_C - b_prime = ell(s_D | p) - ell_C(s_D | p)`.

The selection score is the implied percent reduction in deceptive sentence probability:
`100 * (1 - exp(-Delta(C)))`. A top-K sweep selects the smallest circuit whose
percent reduction reaches `ATTR_PATCH_PERCENT_REDUCTION_THRESHOLD`, or the first
post-flattening K if the threshold is not reached.

After selecting K, compare the discovered circuit against five controls: same-size random
heads, layer-matched random heads, shuffled truthful donors, shuffled deceptive donors, and
wrong-direction patching.


In [ ]:
CIRCUIT_SELECT = os.environ.get("ATTR_PATCH_CIRCUIT_SELECT", "positive").strip().lower()
PERCENT_REDUCTION_THRESHOLD = float(os.environ.get("ATTR_PATCH_PERCENT_REDUCTION_THRESHOLD", "50.0"))
MAX_CIRCUIT_SIZE_RAW = os.environ.get("ATTR_PATCH_CIRCUIT_MAX_SIZE", "auto").strip().lower()
CONTROL_CIRCUIT_COUNT = int(
    os.environ.get(
        "ATTR_PATCH_CONTROL_CIRCUIT_COUNT",
        os.environ.get("ATTR_PATCH_RANDOM_CIRCUIT_COUNT", "8"),
    )
)
VALIDATION_MAX_CHUNKS = int(os.environ.get("ATTR_PATCH_VALIDATION_MAX_CHUNKS", "0"))
VALIDATION_SEED = int(os.environ.get("ATTR_PATCH_VALIDATION_SEED", "17"))
ACTIVATION_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if SENTENCE_SCORE_MODE != "mean_logprob":
    display(
        Markdown(
            "**Note:** percent probability reduction assumes `SENTENCE_SCORE_MODE='mean_logprob'`. "
            f"Current mode is `{SENTENCE_SCORE_MODE}`."
        )
    )

site_df = pd.DataFrame(
    [
        {
            "layer": int(layer_idx),
            "head": int(head_idx),
            "attribution": float(patching_results[layer_idx, head_idx]),
            "abs_attribution": abs(float(patching_results[layer_idx, head_idx])),
        }
        for layer_idx in range(n_layers)
        for head_idx in range(n_heads)
    ]
)

if CIRCUIT_SELECT == "positive" and (site_df["attribution"] > 0).any():
    ranked_site_df = (
        site_df[site_df["attribution"] > 0]
        .sort_values("attribution", ascending=False)
        .reset_index(drop=True)
    )
elif CIRCUIT_SELECT in {"positive", "abs"}:
    ranked_site_df = (
        site_df.sort_values("abs_attribution", ascending=False)
        .reset_index(drop=True)
    )
else:
    raise ValueError("ATTR_PATCH_CIRCUIT_SELECT must be 'positive' or 'abs'.")

if MAX_CIRCUIT_SIZE_RAW in {"", "auto", "all", "none"}:
    max_circuit_size = len(ranked_site_df)
else:
    max_circuit_size = min(int(MAX_CIRCUIT_SIZE_RAW), len(ranked_site_df))
if max_circuit_size <= 0:
    raise ValueError("No ranked circuit sites are available.")

ranked_site_df = ranked_site_df.head(max_circuit_size).reset_index(drop=True)
ranked_sites = [
    (int(row.layer), int(row.head))
    for row in ranked_site_df.itertuples(index=False)
]
all_sites = [(layer_idx, head_idx) for layer_idx in range(n_layers) for head_idx in range(n_heads)]
rng = np.random.default_rng(VALIDATION_SEED)

display(ranked_site_df.head(20))
print(
    f"Ranked {len(ranked_sites)} candidate head-site(s) using CIRCUIT_SELECT={CIRCUIT_SELECT!r}; "
    f"target percent reduction >= {PERCENT_REDUCTION_THRESHOLD:.2f}% | "
    f"patch_scope={PATCH_SCOPE!r}."
)
print(
    f"Candidate search space: {len(ranked_sites)} ranked site(s) out of "
    f"{len(all_sites)} total attention-head site(s)."
)
if CIRCUIT_SELECT == "positive":
    print("Note: top_k with CIRCUIT_SELECT='positive' is not the full attention-head set.")


def normalize_sites(selected_sites):
    return [(int(layer_idx), int(head_idx)) for layer_idx, head_idx in selected_sites]


def group_sites_by_layer(selected_sites):
    grouped = {}
    for layer_idx, head_idx in normalize_sites(selected_sites):
        grouped.setdefault(layer_idx, set()).add(head_idx)
    return {layer_idx: sorted(heads) for layer_idx, heads in grouped.items()}


def validation_chunks(chunks):
    if VALIDATION_MAX_CHUNKS > 0:
        return chunks[:VALIDATION_MAX_CHUNKS]
    return chunks


def percent_probability_reduction_from_delta(delta):
    return 100.0 * (1.0 - math.exp(-float(delta)))


def score_unpatched_chunks(chunks, *, target_role="deceptive"):
    scores = []
    for chunk_pairs in chunks:
        _, target_batch, _, target_inputs = build_source_target_batches(
            chunk_pairs,
            chunk_pairs,
            source_role="truthful",
            target_role=target_role,
        )
        with torch.inference_mode():
            logits = model.trace(target_inputs, trace=False).logits
        metric = target_metric_from_logits(logits, target_batch)
        scores.append(float(metric.item()))
        del logits, metric, target_batch, target_inputs
        clear_memory()
    return scores


def weighted_average_chunk_scores(chunks, scores):
    weighted_total = 0.0
    weight_total = 0
    for chunk_pairs, score in zip(chunks, scores):
        weight = len(chunk_pairs)
        weighted_total += float(score) * weight
        weight_total += weight
    return weighted_total / float(weight_total)


def score_source_target_baselines(chunks, *, return_chunk_records=False):
    source_score_total = 0.0
    target_score_total = 0.0
    target_metric_total = 0.0
    truth_minus_total = 0.0
    weight_total = 0
    chunk_records = []
    for chunk_idx, chunk_pairs in enumerate(chunks):
        source_batch, target_batch, source_inputs, target_inputs = build_batches_for_pairs(chunk_pairs)
        weight = len(chunk_pairs)
        with torch.inference_mode():
            source_logits = model.trace(source_inputs, trace=False).logits
        source_scores = sentence_score_by_row(source_logits, source_batch).detach().cpu()
        source_score = float(source_scores.mean().item())
        del source_logits
        clear_memory()

        with torch.inference_mode():
            target_logits = model.trace(target_inputs, trace=False).logits
        target_scores = sentence_score_by_row(target_logits, target_batch).detach().cpu()
        target_score = float(target_scores.mean().item())
        target_metric = float((-target_scores).mean().item())
        truth_minus = float((source_scores - target_scores).mean().item())
        del target_logits
        clear_memory()

        source_score_total += source_score * weight
        target_score_total += target_score * weight
        target_metric_total += target_metric * weight
        truth_minus_total += truth_minus * weight
        weight_total += weight
        chunk_records.append(
            {
                "chunk_index": int(chunk_idx),
                "n_pairs": int(weight),
                "source_truthful_score": source_score,
                "target_deceptive_score": target_score,
                "target_metric": target_metric,
                "truth_minus_deceptive_metric": truth_minus,
            }
        )
        del source_scores, target_scores, source_batch, target_batch, source_inputs, target_inputs
        clear_memory()

    source_truthful_score = source_score_total / weight_total
    target_deceptive_score = target_score_total / weight_total
    target_baseline = target_metric_total / weight_total
    truth_minus_deceptive = truth_minus_total / weight_total
    if return_chunk_records:
        return source_truthful_score, target_deceptive_score, target_baseline, truth_minus_deceptive, chunk_records
    return source_truthful_score, target_deceptive_score, target_baseline, truth_minus_deceptive


def shuffled_donor_pairs_for_chunk(chunk_pairs, donor_pool, *, rng):
    donor_pool = list(donor_pool)
    shuffled = []
    for target_pair in chunk_pairs:
        target_pair_index = int(target_pair["pair_index"])
        eligible = [
            donor_pair for donor_pair in donor_pool
            if int(donor_pair["pair_index"]) != target_pair_index
        ]
        if not eligible:
            eligible = donor_pool
        shuffled.append(eligible[int(rng.integers(len(eligible)))])
    return shuffled


def patch_circuit_chunk(
    chunk_pairs,
    selected_sites,
    *,
    source_pairs=None,
    source_role="truthful",
    target_role="deceptive",
):
    grouped = group_sites_by_layer(selected_sites)
    if not grouped:
        raise ValueError("No circuit sites were selected.")
    if source_pairs is None:
        source_pairs = chunk_pairs

    source_batch, target_batch, source_inputs, target_inputs = build_source_target_batches(
        source_pairs,
        chunk_pairs,
        source_role=source_role,
        target_role=target_role,
    )
    source_rows = list(source_batch["rows"])
    target_rows = list(target_batch["rows"])
    source_proxies = {}
    with torch.inference_mode():
        with model.trace(source_inputs):
            for layer_idx in grouped:
                source_proxies[layer_idx] = attn_out_input(layers[layer_idx]).save()

    source_tensors = {
        layer_idx: saved_value(proxy).detach().to("cpu")
        for layer_idx, proxy in source_proxies.items()
    }
    del source_proxies, source_batch, source_inputs
    clear_memory()

    device_sources = {
        layer_idx: tensor.to(ACTIVATION_DEVICE)
        for layer_idx, tensor in source_tensors.items()
    }
    with torch.inference_mode():
        with model.trace(target_inputs):
            for layer_idx, heads in grouped.items():
                current = attn_out_input(layers[layer_idx])
                source = device_sources[layer_idx]
                apply_scoped_head_patch(current, source, source_rows, target_rows, heads)
            logits = model.lm_head.output
            metric = target_metric_from_logits(logits, target_batch).save()

    value = float(saved_value(metric).item())
    del metric, target_batch, target_inputs, source_tensors, device_sources
    clear_memory()
    return value


def score_circuit_on_chunks(
    selected_sites,
    chunks,
    *,
    target_baseline,
    unpatched_scores=None,
    source_pair_fn=None,
    source_role="truthful",
    target_role="deceptive",
    label="circuit",
):
    chunks = validation_chunks(chunks)
    if unpatched_scores is None:
        unpatched_scores = score_unpatched_chunks(chunks)
    patched_total = 0.0
    unpatched_total = 0.0
    weight_total = 0
    for chunk_idx, chunk_pairs in enumerate(chunks):
        source_pairs = None if source_pair_fn is None else source_pair_fn(chunk_pairs)
        patched = patch_circuit_chunk(
            chunk_pairs,
            selected_sites,
            source_pairs=source_pairs,
            source_role=source_role,
            target_role=target_role,
        )
        unpatched = float(unpatched_scores[chunk_idx])
        weight = len(chunk_pairs)
        patched_total += patched * weight
        unpatched_total += unpatched * weight
        weight_total += weight
        print(
            f"{label} | chunk {chunk_idx + 1}/{len(chunks)} | "
            f"b_prime={unpatched:.4f} | m={patched:.4f}"
        )
    patched_metric = patched_total / weight_total
    unpatched_metric = unpatched_total / weight_total
    delta = patched_metric - float(target_baseline)
    percent_reduction = percent_probability_reduction_from_delta(delta)
    return {
        "unpatched_metric": unpatched_metric,
        "patched_metric": patched_metric,
        "source_role": source_role,
        "target_role": target_role,
        "target_baseline_metric": float(target_baseline),
        "delta": delta,
        "percent_probability_reduction": percent_reduction,
        "unpatched_target_sentence_score": -unpatched_metric,
        "patched_target_sentence_score": -patched_metric,
        "target_sentence_score_delta": (-patched_metric) - (-unpatched_metric),
        "unpatched_target_deceptive_score": -unpatched_metric,
        "patched_target_deceptive_score": -patched_metric,
        "target_deceptive_score_delta": (-patched_metric) - (-unpatched_metric),
        "n_chunks": len(chunks),
        "n_pairs": int(sum(len(chunk) for chunk in chunks)),
        "circuit_size": len(normalize_sites(selected_sites)),
    }


def top_ranked_sites(edge_count):
    edge_count = int(edge_count)
    if edge_count <= 0:
        raise ValueError("edge_count must be positive.")
    if edge_count > len(ranked_sites):
        raise ValueError(f"edge_count={edge_count} exceeds {len(ranked_sites)} ranked sites.")
    return ranked_sites[:edge_count]


def random_sites_for_size(edge_count, *, rng):
    random_site_indices = rng.choice(len(all_sites), size=int(edge_count), replace=False)
    return [all_sites[int(idx)] for idx in random_site_indices]


def layer_matched_random_sites(selected_sites, *, rng):
    selected_sites = normalize_sites(selected_sites)
    selected_set = set(selected_sites)
    sampled_sites = []
    for layer_idx, heads in group_sites_by_layer(selected_sites).items():
        layer_candidates = [
            (int(layer_idx), int(head_idx))
            for head_idx in range(n_heads)
            if (int(layer_idx), int(head_idx)) not in selected_set
        ]
        if len(layer_candidates) < len(heads):
            layer_candidates = [(int(layer_idx), int(head_idx)) for head_idx in range(n_heads)]
        sampled_indices = rng.choice(len(layer_candidates), size=len(heads), replace=False)
        sampled_sites.extend(layer_candidates[int(idx)] for idx in sampled_indices)
    return sampled_sites


def candidate_circuit_sizes(ranked_site_count):
    candidate_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256]
    return [k for k in candidate_sizes if k <= int(ranked_site_count)]


def evaluate_top_k_for_percent_reduction(
    edge_count,
    chunks,
    *,
    target_baseline,
    unpatched_scores,
):
    edge_count = int(edge_count)
    record = score_circuit_on_chunks(
        top_ranked_sites(edge_count),
        chunks,
        target_baseline=target_baseline,
        unpatched_scores=unpatched_scores,
        label=f"top_{edge_count}",
    )
    record = {
        "candidate_edge_count": edge_count,
        "meets_threshold": bool(
            record["percent_probability_reduction"] >= float(PERCENT_REDUCTION_THRESHOLD)
            and record["delta"] > 0.0
        ),
        **record,
    }
    print(
        f"top_{edge_count} aggregate | delta={record['delta']:.4f} | "
        f"percent_reduction={record['percent_probability_reduction']:.2f}%"
    )
    return record


def choose_top_k_from_sweep(sweep_df):
    meeting_df = sweep_df[sweep_df["meets_threshold"]].copy()
    if not meeting_df.empty:
        best_row = meeting_df.sort_values("candidate_edge_count").iloc[0]
        return int(best_row["candidate_edge_count"]), best_row.to_dict(), "threshold_reached"

    ordered = sweep_df.sort_values("candidate_edge_count").reset_index(drop=True)
    flatten_tol = float(os.environ.get("ATTR_PATCH_FLATTENING_PERCENT_POINT_TOL", "1.0"))
    for idx in range(len(ordered) - 1):
        current = float(ordered.loc[idx, "percent_probability_reduction"])
        nxt = float(ordered.loc[idx + 1, "percent_probability_reduction"])
        if current > 0.0 and (nxt - current) <= flatten_tol:
            row = ordered.loc[idx]
            return int(row["candidate_edge_count"]), row.to_dict(), "threshold_not_reached_flattened"

    best_row = ordered.loc[ordered["percent_probability_reduction"].idxmax()]
    return int(best_row["candidate_edge_count"]), best_row.to_dict(), "threshold_not_reached_best_percent"


def sweep_top_k_percent_reduction_circuits(
    chunks,
    *,
    target_baseline,
    unpatched_scores,
    candidate_sizes=None,
):
    if candidate_sizes is None:
        candidate_sizes = candidate_circuit_sizes(len(ranked_sites))
    records = []
    for edge_count in candidate_sizes:
        records.append(
            evaluate_top_k_for_percent_reduction(
                edge_count,
                chunks,
                target_baseline=target_baseline,
                unpatched_scores=unpatched_scores,
            )
        )
    sweep_df = pd.DataFrame(records).reset_index(drop=True)
    discovered_edge_count, discovered_record, status = choose_top_k_from_sweep(sweep_df)
    return discovered_edge_count, discovered_record, sweep_df, status

def score_selected_circuit_controls(
    selected_sites,
    chunks,
    *,
    target_baseline,
    unpatched_scores,
    rng,
    donor_pool,
    control_count=CONTROL_CIRCUIT_COUNT,
    label_prefix="control",
):
    records = []
    selected_sites = normalize_sites(selected_sites)
    chunks = validation_chunks(chunks)

    wrong_direction_unpatched_scores = score_unpatched_chunks(chunks, target_role="truthful")
    wrong_direction_target_baseline = weighted_average_chunk_scores(
        chunks,
        wrong_direction_unpatched_scores,
    )

    control_specs = [
        "random",
        "layer_matched_random",
        "shuffled_truthful_donor",
        "shuffled_deceptive_donor",
        "wrong_direction",
    ]
    for control_kind in control_specs:
        for control_idx in range(int(control_count)):
            control_target_baseline = target_baseline
            control_unpatched_scores = unpatched_scores
            source_role = "truthful"
            target_role = "deceptive"
            source_pair_fn = None

            if control_kind == "random":
                control_sites = random_sites_for_size(len(selected_sites), rng=rng)
            elif control_kind == "layer_matched_random":
                control_sites = layer_matched_random_sites(selected_sites, rng=rng)
            elif control_kind == "shuffled_truthful_donor":
                control_sites = selected_sites
                source_pair_fn = lambda chunk_pairs, rng=rng: shuffled_donor_pairs_for_chunk(
                    chunk_pairs,
                    donor_pool,
                    rng=rng,
                )
            elif control_kind == "shuffled_deceptive_donor":
                control_sites = selected_sites
                source_role = "deceptive"
                source_pair_fn = lambda chunk_pairs, rng=rng: shuffled_donor_pairs_for_chunk(
                    chunk_pairs,
                    donor_pool,
                    rng=rng,
                )
            elif control_kind == "wrong_direction":
                control_sites = selected_sites
                source_role = "deceptive"
                target_role = "truthful"
                control_target_baseline = wrong_direction_target_baseline
                control_unpatched_scores = wrong_direction_unpatched_scores
            else:
                raise ValueError(f"Unsupported control_kind={control_kind!r}.")

            record = score_circuit_on_chunks(
                control_sites,
                chunks,
                target_baseline=control_target_baseline,
                unpatched_scores=control_unpatched_scores,
                source_pair_fn=source_pair_fn,
                source_role=source_role,
                target_role=target_role,
                label=f"{label_prefix}_{control_kind}_{control_idx:02d}",
            )
            records.append(
                {
                    "circuit_kind": control_kind,
                    "circuit_id": f"{label_prefix}_{control_kind}_{control_idx:02d}",
                    "control_index": int(control_idx),
                    "candidate_edge_count": len(selected_sites),
                    **record,
                }
            )
    return pd.DataFrame(records)


In [ ]:
baseline_chunks = validation_chunks(pair_chunks)
(
    VALIDATION_SOURCE_TRUTHFUL_SCORE,
    VALIDATION_TARGET_DECEPTIVE_SCORE,
    VALIDATION_TARGET_BASELINE,
    VALIDATION_TRUTH_MINUS_DECEPTIVE_BASELINE,
    validation_baseline_chunk_records,
) = score_source_target_baselines(
    baseline_chunks,
    return_chunk_records=True,
)
baseline_unpatched_scores = [
    float(record["target_metric"])
    for record in validation_baseline_chunk_records
]
validation_baseline_df = pd.DataFrame(validation_baseline_chunk_records)
display(validation_baseline_df)
print(f"Validation fixed truthful source score: {VALIDATION_SOURCE_TRUTHFUL_SCORE:.4f}")
print(f"Validation deceptive target score: {VALIDATION_TARGET_DECEPTIVE_SCORE:.4f}")
print(f"Validation target baseline b_prime = -score_D: {VALIDATION_TARGET_BASELINE:.4f}")
print(f"Validation truth-minus-deceptive reference: {VALIDATION_TRUTH_MINUS_DECEPTIVE_BASELINE:.4f}")
print(f"Percent reduction threshold: {PERCENT_REDUCTION_THRESHOLD:.2f}%")

candidate_sizes = candidate_circuit_sizes(len(ranked_sites))
print(f"Sweeping candidate circuit sizes: {candidate_sizes}")
discovered_edge_count, discovered_record, circuit_search_df, circuit_search_status = (
    sweep_top_k_percent_reduction_circuits(
        baseline_chunks,
        target_baseline=VALIDATION_TARGET_BASELINE,
        unpatched_scores=baseline_unpatched_scores,
        candidate_sizes=candidate_sizes,
    )
)
discovered_sites = top_ranked_sites(discovered_edge_count)
discovered_circuit_df = ranked_site_df.head(discovered_edge_count).reset_index(drop=True)

display(circuit_search_df)
display(discovered_circuit_df)
print(
    f"Selected {discovered_edge_count} head-site(s): {circuit_search_status}; "
    f"delta={discovered_record['delta']:.4f}; "
    f"percent_reduction={discovered_record['percent_probability_reduction']:.2f}%; "
    f"threshold={PERCENT_REDUCTION_THRESHOLD:.2f}%."
)

selected_controls_df = score_selected_circuit_controls(
    discovered_sites,
    baseline_chunks,
    target_baseline=VALIDATION_TARGET_BASELINE,
    unpatched_scores=baseline_unpatched_scores,
    rng=rng,
    donor_pool=prepared_pairs,
    control_count=CONTROL_CIRCUIT_COUNT,
    label_prefix=f"top_{discovered_edge_count}",
)

comparison_records = [
    {
        "circuit_kind": "target_reference",
        "circuit_id": "target_unpatched_b_prime",
        "search_status": "reference",
        "percent_reduction_threshold": PERCENT_REDUCTION_THRESHOLD,
        "patch_scope": PATCH_SCOPE,
        "candidate_edge_count": 0,
        "meets_threshold": False,
        "ranked_candidate_count": len(ranked_sites),
        "total_attention_sites": len(all_sites),
        "source_truthful_score": VALIDATION_SOURCE_TRUTHFUL_SCORE,
        "truth_minus_deceptive_reference": VALIDATION_TRUTH_MINUS_DECEPTIVE_BASELINE,
        "unpatched_metric": VALIDATION_TARGET_BASELINE,
        "patched_metric": VALIDATION_TARGET_BASELINE,
        "source_role": "truthful",
        "target_role": "deceptive",
        "target_baseline_metric": VALIDATION_TARGET_BASELINE,
        "delta": 0.0,
        "percent_probability_reduction": 0.0,
        "unpatched_target_sentence_score": VALIDATION_TARGET_DECEPTIVE_SCORE,
        "patched_target_sentence_score": VALIDATION_TARGET_DECEPTIVE_SCORE,
        "target_sentence_score_delta": 0.0,
        "unpatched_target_deceptive_score": VALIDATION_TARGET_DECEPTIVE_SCORE,
        "patched_target_deceptive_score": VALIDATION_TARGET_DECEPTIVE_SCORE,
        "target_deceptive_score_delta": 0.0,
        "n_chunks": len(baseline_chunks),
        "n_pairs": int(sum(len(chunk) for chunk in baseline_chunks)),
        "circuit_size": 0,
    },
    {
        "circuit_kind": "discovered",
        "circuit_id": f"top_{discovered_edge_count}",
        "search_status": circuit_search_status,
        "percent_reduction_threshold": PERCENT_REDUCTION_THRESHOLD,
        "patch_scope": PATCH_SCOPE,
        "ranked_candidate_count": len(ranked_sites),
        "total_attention_sites": len(all_sites),
        "source_truthful_score": VALIDATION_SOURCE_TRUTHFUL_SCORE,
        "truth_minus_deceptive_reference": VALIDATION_TRUTH_MINUS_DECEPTIVE_BASELINE,
        **discovered_record,
    },
]
for row in selected_controls_df.to_dict(orient="records"):
    comparison_records.append(
        {
            "search_status": "selected_size_control",
            "percent_reduction_threshold": PERCENT_REDUCTION_THRESHOLD,
            "patch_scope": PATCH_SCOPE,
            "ranked_candidate_count": len(ranked_sites),
            "total_attention_sites": len(all_sites),
            "source_truthful_score": VALIDATION_SOURCE_TRUTHFUL_SCORE,
            "truth_minus_deceptive_reference": VALIDATION_TRUTH_MINUS_DECEPTIVE_BASELINE,
            "meets_threshold": False,
            **row,
        }
    )

control_comparison_df = pd.DataFrame(comparison_records)
display(control_comparison_df)
summary_comparison_df = control_comparison_df[
    control_comparison_df["circuit_kind"].isin(
        [
            "discovered",
            "random",
            "layer_matched_random",
            "shuffled_truthful_donor",
            "shuffled_deceptive_donor",
            "wrong_direction",
        ]
    )
].copy()
display(
    summary_comparison_df.groupby("circuit_kind", as_index=False)[
        ["delta", "percent_probability_reduction"]
    ]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)
fig = px.strip(
    summary_comparison_df,
    x="circuit_kind",
    y="percent_probability_reduction",
    color="circuit_kind",
    title="Percent reduction in deceptive sentence probability: circuit vs controls",
)
fig.show()


In [ ]:
CROSS_CORPUS_ENVS = [
    env.strip()
    for env in os.environ.get(
        "ATTR_PATCH_CROSS_CORPUS_ENVS",
        "advisor_audit,car_sales,gridworld,interview",
    ).split(",")
    if env.strip()
]
CROSS_CORPUS_PAIR_COUNT = int(os.environ.get("ATTR_PATCH_CROSS_CORPUS_PAIR_COUNT", "3"))
CROSS_CORPUS_SEARCH_LIMIT = int(os.environ.get("ATTR_PATCH_CROSS_CORPUS_SEARCH_LIMIT", "128"))


def count_alpha_words(text):
    return len(re.findall(r"[A-Za-z]+", str(text)))


def usable_sentence(text):
    text = str(text).strip()
    if not text:
        return False
    if EXCLUDE_MULTILINE_SENTENCES and "\n" in text:
        return False
    if MIN_SENTENCE_ALPHA_WORDS > 0 and count_alpha_words(text) < MIN_SENTENCE_ALPHA_WORDS:
        return False
    return True


def choose_generic_truthful_donor(shared_entry, target_sentence):
    target_norm = ap.normalize_sentence_for_compare(target_sentence)
    candidates = []
    for gen_idx, generation in enumerate(shared_entry.get("generations") or []):
        if generation.get("is_truthful") is not True:
            continue
        first_sentence, _ = ap.extract_first_sentence(generation.get("gen_text", ""))
        first_sentence = str(first_sentence).strip()
        if not usable_sentence(first_sentence):
            continue
        if ap.normalize_sentence_for_compare(first_sentence) == target_norm:
            continue
        candidates.append(
            {
                "gen_idx": int(gen_idx),
                "first_sentence": first_sentence,
                "prompt": generation.get("prompt", shared_entry.get("prompt", "")),
                "prefix_text": generation.get("prefix_text", shared_entry.get("prefix_text", "")),
                "full_generation_text": generation.get("full_generation_text", ""),
                "is_truthful": generation.get("is_truthful"),
                "deceptive": generation.get("deceptive"),
                "parse_error": generation.get("parse_error"),
                "evaluation": generation.get("evaluation"),
                "word_count": count_alpha_words(first_sentence),
            }
        )
    if not candidates:
        return None, []
    candidates = sorted(candidates, key=lambda row: (abs(row["word_count"] - 12), len(row["first_sentence"])))
    return candidates[0], candidates


def load_generic_commitment_pairs_for_env(env_name, *, pair_count, search_limit):
    localization_dir = REPO_ROOT / "DatasetMain" / env_name / MODEL_ID / "localization"
    if not localization_dir.exists():
        print(f"Skipping {env_name}: missing {localization_dir}")
        return pd.DataFrame()

    rows = []
    paths = sorted(localization_dir.glob("sentence_localization_*.json"))
    for path in paths:
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        history = payload.get("history") or []
        if len(history) < 2:
            continue
        example_id = str(payload.get("example_id", path.stem))
        for right_pos in range(1, len(history)):
            left_pos = right_pos - 1
            shared_entry = history[left_pos]
            target_entry = history[right_pos]
            try:
                shared_rate = float(shared_entry.get("deception_rate", float("nan")))
                target_rate = float(target_entry.get("deception_rate", float("nan")))
                commitment_delta = target_rate - shared_rate
            except Exception:
                continue
            if not math.isfinite(commitment_delta) or commitment_delta <= MIN_COMMITMENT_DELTA:
                continue
            if target_rate < MIN_COMMITMENT_DECEPTION_RATE:
                continue

            shared_num_valid = int(shared_entry.get("num_valid") or 0)
            target_num_valid = int(target_entry.get("num_valid") or 0)
            if MIN_NUM_VALID > 0 and (
                shared_num_valid < MIN_NUM_VALID or target_num_valid < MIN_NUM_VALID
            ):
                continue

            deceptive_sentence = str(target_entry.get("sentence_text", "")).strip()
            if not usable_sentence(deceptive_sentence):
                continue
            donor_row, donor_candidates = choose_generic_truthful_donor(shared_entry, deceptive_sentence)
            if donor_row is None:
                continue

            prompt = str(target_entry.get("prompt", payload.get("prompt", "")))
            shared_context_text = str(shared_entry.get("prefix_text", ""))
            truthful_sentence = str(donor_row["first_sentence"]).strip()
            if not prompt or not shared_context_text:
                continue

            rows.append(
                {
                    "pair_id": (
                        f"{env_name}__{ap.slugify(example_id)}__sent_{int(right_pos)}__"
                        f"donor_{int(donor_row['gen_idx'])}"
                    ),
                    "localization_path": str(path),
                    "example_id": example_id,
                    "shared_context_sentence_pos": int(left_pos),
                    "commitment_sentence_pos": int(right_pos),
                    "shared_context_sentence_text": str(shared_entry.get("sentence_text", "")),
                    "deceptive_commitment_sentence": deceptive_sentence,
                    "truthful_donor_sentence": truthful_sentence,
                    "prompt": prompt,
                    "shared_context_text": shared_context_text,
                    "shared_context_deception_rate": shared_rate,
                    "deceptive_prefix_deception_rate": target_rate,
                    "commitment_deception_rate": target_rate,
                    "commitment_delta": commitment_delta,
                    "shared_context_num_valid": shared_num_valid,
                    "shared_context_num_truthful": shared_entry.get("num_truthful"),
                    "deceptive_prefix_num_valid": target_num_valid,
                    "deceptive_prefix_num_truthful": target_entry.get("num_truthful"),
                    "donor_generation_idx": int(donor_row["gen_idx"]),
                    "donor_full_generation_text": donor_row.get("full_generation_text", ""),
                    "donor_is_truthful": donor_row.get("is_truthful") is True,
                    "donor_deceptive": donor_row.get("deceptive"),
                    "donor_parse_error": donor_row.get("parse_error"),
                    "donor_evaluation": ap.to_json_safe(donor_row.get("evaluation")),
                    "donor_clarity_score": float(donor_row["word_count"]),
                    "n_truthful_donors": int(len(donor_candidates)),
                    "shared_prefix_text": prompt + shared_context_text,
                    "deceptive_branch_text": prompt + ap.append_continuation(
                        shared_context_text,
                        deceptive_sentence,
                    ),
                    "truthful_branch_text": prompt + ap.append_continuation(
                        shared_context_text,
                        truthful_sentence,
                    ),
                }
            )
            if len(rows) > max(search_limit * 4, search_limit + 100):
                rows = sorted(
                    rows,
                    key=lambda row: (
                        row["commitment_delta"],
                        row["deceptive_prefix_deception_rate"],
                        row["n_truthful_donors"],
                        row["donor_clarity_score"],
                    ),
                    reverse=True,
                )[:search_limit]

    if not rows:
        return pd.DataFrame()
    rows = sorted(
        rows,
        key=lambda row: (
            row["commitment_delta"],
            row["deceptive_prefix_deception_rate"],
            row["n_truthful_donors"],
            row["donor_clarity_score"],
        ),
        reverse=True,
    )[:pair_count]
    df = pd.DataFrame(rows).reset_index(drop=True)
    df.insert(0, "pair_index", range(len(df)))
    return df


def prepare_records_without_length_cap(pairs):
    global MAX_INPUT_TOKENS
    previous_max_input_tokens = MAX_INPUT_TOKENS
    try:
        MAX_INPUT_TOKENS = None
        return prepare_pair_records(pairs)
    finally:
        MAX_INPUT_TOKENS = previous_max_input_tokens


cross_records = []
for env_name in CROSS_CORPUS_ENVS:
    env_pairs_df = load_generic_commitment_pairs_for_env(
        env_name,
        pair_count=CROSS_CORPUS_PAIR_COUNT,
        search_limit=CROSS_CORPUS_SEARCH_LIMIT,
    )
    if env_pairs_df.empty:
        cross_records.append(
            {
                "environment": env_name,
                "status": "no_pairs",
                "n_pairs": 0,
            }
        )
        continue

    env_prepared_pairs = prepare_records_without_length_cap(env_pairs_df)
    env_pair_chunks = validation_chunks(pair_chunk_slices(env_prepared_pairs, BATCH_PAIR_COUNT))
    (
        env_source_truthful_score,
        env_target_deceptive_score,
        env_target_baseline,
        env_truth_minus_deceptive,
        env_baseline_chunk_records,
    ) = score_source_target_baselines(
        env_pair_chunks,
        return_chunk_records=True,
    )
    env_unpatched_scores = [
        float(record["target_metric"])
        for record in env_baseline_chunk_records
    ]
    env_record = score_circuit_on_chunks(
        discovered_sites,
        env_pair_chunks,
        target_baseline=env_target_baseline,
        unpatched_scores=env_unpatched_scores,
        label=f"{env_name}_discovered",
    )
    cross_records.append(
        {
            "environment": env_name,
            "status": "ok",
            "circuit_kind": "discovered",
            "circuit_id": f"{env_name}_discovered",
            "source_truthful_score": env_source_truthful_score,
            "target_deceptive_score": env_target_deceptive_score,
            "target_baseline": env_target_baseline,
            "truth_minus_deceptive_reference": env_truth_minus_deceptive,
            "max_total_len": max(int(pair["max_total_len"]) for pair in env_prepared_pairs),
            **env_record,
        }
    )

    env_controls_df = score_selected_circuit_controls(
        discovered_sites,
        env_pair_chunks,
        target_baseline=env_target_baseline,
        unpatched_scores=env_unpatched_scores,
        rng=rng,
        donor_pool=env_prepared_pairs,
        control_count=CONTROL_CIRCUIT_COUNT,
        label_prefix=env_name,
    )
    for control_row in env_controls_df.to_dict(orient="records"):
        cross_records.append(
            {
                "environment": env_name,
                "status": "ok",
                "source_truthful_score": env_source_truthful_score,
                "target_deceptive_score": env_target_deceptive_score,
                "target_baseline": env_target_baseline,
                "truth_minus_deceptive_reference": env_truth_minus_deceptive,
                "max_total_len": max(int(pair["max_total_len"]) for pair in env_prepared_pairs),
                **control_row,
            }
        )
    clear_memory()

cross_corpus_df = pd.DataFrame(cross_records)
display(cross_corpus_df)


## Circuit steering

Attribution patching tells us *where* to intervene. The matched truthful-minus-deceptive
activation differences tell us *which direction* to intervene in.

For each selected head `h`, estimate `d_h = E[z_h_truthful - z_h_deceptive]` on the
matched pairs. During generation, add `alpha * d_h` at the selected head slices, so the
steering intervention is reusable and does not need a donor example.


In [ ]:
STEERING_ALPHA = float(os.environ.get("ATTR_PATCH_STEERING_ALPHA", "1.0"))
STEERING_GENERATION_COUNT = int(os.environ.get("ATTR_PATCH_STEERING_GENERATION_COUNT", "10"))
STEERING_MAX_NEW_TOKENS = int(os.environ.get("ATTR_PATCH_STEERING_MAX_NEW_TOKENS", "96"))
STEERING_TEMPERATURE = float(os.environ.get("ATTR_PATCH_STEERING_TEMPERATURE", "0.7"))
STEERING_TOP_P = float(os.environ.get("ATTR_PATCH_STEERING_TOP_P", "0.95"))
STEERING_SEED = int(os.environ.get("ATTR_PATCH_STEERING_SEED", "23"))
STEERING_INCLUDE_BASELINE = os.environ.get("ATTR_PATCH_STEERING_INCLUDE_BASELINE", "0") == "1"
STEERING_POSITION = os.environ.get("ATTR_PATCH_STEERING_POSITION", "last").strip().lower()

if STEERING_POSITION not in {"last", "all"}:
    raise ValueError("ATTR_PATCH_STEERING_POSITION must be 'last' or 'all'.")


def compute_head_steering_vectors(chunks, selected_sites):
    grouped = group_sites_by_layer(selected_sites)
    sums = {
        (layer_idx, head_idx): torch.zeros(head_dim, dtype=torch.float32)
        for layer_idx, heads in grouped.items()
        for head_idx in heads
    }
    counts = {site: 0 for site in sums}

    for layer_idx, heads in grouped.items():
        print(f"Computing steering directions for layer {layer_idx:02d} ({len(heads)} head(s))")
        for chunk_pairs in chunks:
            source_batch, target_batch, source_inputs, target_inputs = build_batches_for_pairs(chunk_pairs)
            with torch.inference_mode():
                with model.trace(source_inputs):
                    source_proxy = attn_out_input(layers[layer_idx]).save()
            source_acts = saved_value(source_proxy).detach().float().cpu()
            del source_proxy
            clear_memory()

            with torch.inference_mode():
                with model.trace(target_inputs):
                    target_proxy = attn_out_input(layers[layer_idx]).save()
            target_acts = saved_value(target_proxy).detach().float().cpu()

            for local_pair_idx, _pair in enumerate(chunk_pairs):
                source_row = source_batch["rows"][local_pair_idx]
                target_row = target_batch["rows"][local_pair_idx]
                source_slice = slice(
                    int(source_row["score_start_pos"]),
                    int(source_row["score_stop_pos"]),
                )
                target_slice = slice(
                    int(target_row["score_start_pos"]),
                    int(target_row["score_stop_pos"]),
                )
                for head_idx in heads:
                    start = int(head_idx) * head_dim
                    stop = start + head_dim
                    source_vec = source_acts[
                        local_pair_idx,
                        source_slice,
                        start:stop,
                    ].mean(dim=0)
                    target_vec = target_acts[
                        local_pair_idx,
                        target_slice,
                        start:stop,
                    ].mean(dim=0)
                    site = (int(layer_idx), int(head_idx))
                    sums[site] += source_vec - target_vec
                    counts[site] += 1

            del target_proxy, source_acts, target_acts, source_batch, target_batch, source_inputs, target_inputs
            clear_memory()

    vectors = {
        site: sums[site] / max(int(counts[site]), 1)
        for site in sums
    }
    vector_df = pd.DataFrame(
        [
            {
                "layer": int(layer_idx),
                "head": int(head_idx),
                "n_pairs": int(counts[(layer_idx, head_idx)]),
                "direction_norm": float(vector.norm().item()),
                "direction_mean_abs": float(vector.abs().mean().item()),
            }
            for (layer_idx, head_idx), vector in vectors.items()
        ]
    ).sort_values(["layer", "head"]).reset_index(drop=True)
    return vectors, vector_df


steering_vectors, steering_vector_df = compute_head_steering_vectors(pair_chunks, discovered_sites)
display(steering_vector_df)
print(
    f"Computed {len(steering_vectors)} head steering vector(s) "
    f"for alpha={STEERING_ALPHA:.3f}."
)


In [ ]:
def sample_next_token_from_logits(logits, *, temperature, top_p):
    logits = logits.detach().float().cpu()
    if float(temperature) <= 0:
        return torch.argmax(logits, dim=-1, keepdim=True)

    logits = logits / float(temperature)
    if 0 < float(top_p) < 1:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
        sorted_probs = torch.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        remove_mask = cumulative_probs > float(top_p)
        remove_mask[..., 1:] = remove_mask[..., :-1].clone()
        remove_mask[..., 0] = False
        sorted_logits = sorted_logits.masked_fill(remove_mask, float("-inf"))
        filtered_logits = torch.full_like(logits, float("-inf"))
        logits = filtered_logits.scatter(dim=-1, index=sorted_indices, src=sorted_logits)

    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


def steering_position_slice(current):
    if STEERING_POSITION == "last":
        return (slice(None), slice(-1, None))
    return (slice(None), slice(None))


def generate_with_head_steering(prompt_text, *, seed, do_steer):
    torch.manual_seed(int(seed))
    encoded = encode_text_for_model(tokenizer, prompt_text, max_input_tokens=None)
    current_ids = encoded["input_ids"].clone()
    prompt_token_count = int(current_ids.shape[1])
    generated_ids = []
    ended_with_eos = False

    grouped_vectors = group_sites_by_layer(discovered_sites)
    device_vectors = {
        site: vector.to(ACTIVATION_DEVICE)
        for site, vector in steering_vectors.items()
    }

    for _step in range(STEERING_MAX_NEW_TOKENS):
        inputs = {
            "input_ids": current_ids,
            "attention_mask": torch.ones_like(current_ids),
        }
        if do_steer:
            with torch.inference_mode():
                with model.trace(inputs):
                    for layer_idx, heads in grouped_vectors.items():
                        current = attn_out_input(layers[layer_idx])
                        batch_slice, pos_slice = steering_position_slice(current)
                        for head_idx in heads:
                            site = (int(layer_idx), int(head_idx))
                            start = int(head_idx) * head_dim
                            stop = start + head_dim
                            direction = (
                                float(STEERING_ALPHA)
                                * device_vectors[site].view(1, 1, head_dim)
                            )
                            current[batch_slice, pos_slice, start:stop] = (
                                current[batch_slice, pos_slice, start:stop] + direction
                            )
                    next_logits = model.lm_head.output[:, -1, :].save()
            logits = saved_value(next_logits)
        else:
            with torch.inference_mode():
                logits = model.trace(inputs, trace=False).logits[:, -1, :]

        next_token = sample_next_token_from_logits(
            logits,
            temperature=STEERING_TEMPERATURE,
            top_p=STEERING_TOP_P,
        ).to(dtype=current_ids.dtype)
        token_id = int(next_token.item())
        generated_ids.append(token_id)
        current_ids = torch.cat([current_ids, next_token.cpu()], dim=1)
        del logits, next_token, inputs
        clear_memory()

        if tokenizer.eos_token_id is not None and token_id == int(tokenizer.eos_token_id):
            ended_with_eos = True
            break

    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    full_text = tokenizer.decode(current_ids[0], skip_special_tokens=True)
    return {
        "generated_text": generated_text,
        "full_text": full_text,
        "prompt_token_count": prompt_token_count,
        "n_new_tokens": len(generated_ids),
        "ended_with_eos": ended_with_eos,
        "hit_token_cap": len(generated_ids) >= int(STEERING_MAX_NEW_TOKENS) and not ended_with_eos,
    }


steering_prompt_rows = [
    {
        "pair_index": int(pair["pair_index"]),
        "example_id": str(pair["example_id"]),
        "prompt_text": str(pair["shared_prefix_text"]),
        "deceptive_commitment_sentence": str(pair["deceptive_commitment_sentence"]),
        "truthful_donor_sentence": str(pair["truthful_donor_sentence"]),
    }
    for pair in sorted(prepared_pairs, key=lambda row: int(row["pair_index"]))[:STEERING_GENERATION_COUNT]
]

steering_generation_records = []
for sample_idx, prompt_row in enumerate(steering_prompt_rows):
    seed = STEERING_SEED + sample_idx
    if STEERING_INCLUDE_BASELINE:
        baseline = generate_with_head_steering(
            prompt_row["prompt_text"],
            seed=seed,
            do_steer=False,
        )
        steering_generation_records.append(
            {
                **prompt_row,
                "sample_index": int(sample_idx),
                "condition": "baseline",
                "alpha": 0.0,
                "seed": int(seed),
                **baseline,
            }
        )

    steered = generate_with_head_steering(
        prompt_row["prompt_text"],
        seed=seed,
        do_steer=True,
    )
    steering_generation_records.append(
        {
            **prompt_row,
            "sample_index": int(sample_idx),
            "condition": "steered",
            "alpha": float(STEERING_ALPHA),
            "seed": int(seed),
            **steered,
        }
    )
    print(
        f"Generated {sample_idx + 1}/{len(steering_prompt_rows)} | "
        f"condition=steered | n_new_tokens={steered['n_new_tokens']}"
    )

steering_generations_df = pd.DataFrame(steering_generation_records)
display(
    steering_generations_df[
        [
            "sample_index",
            "condition",
            "alpha",
            "n_new_tokens",
            "hit_token_cap",
            "deceptive_commitment_sentence",
            "truthful_donor_sentence",
            "generated_text",
        ]
    ]
)
